In [ ]:
%%configure -f
{
    "driverCores": 1,
    "driverMemory": "14g",
    "executorCores": 1,
    "executorMemory": "14g",
    "numExecutors": 1
}


In [ ]:
params = None


In [ ]:
from time import sleep
from re import fullmatch
from typing import Union
from pyodbc import connect
from json import loads, dumps
from jsonschema import validate
from delta.tables import DeltaTable
from pyspark.sql.dataframe import DataFrame
from datetime import datetime, time, timezone
from notebookutils.credentials import getSecret
from pyspark.sql import types as PySparkSQLTypes
from requests import Session, post, get, Response
from pyspark.sql.functions import col, lit, sha2, concat_ws, when, expr
from pyspark.sql.types import StructType, StructField, StringType, TimestampType


In [ ]:
class Decorator:

    @classmethod
    def validate_kwargs(cls, schema):
        def _external_wrapper(func):
            def _internal_wrapper(*args, **kwargs):
                validate(instance=kwargs, schema=schema)
                return func(*args, **kwargs)
            _internal_wrapper.__name__ = func.__name__
            return _internal_wrapper
        return _external_wrapper


In [ ]:
class RegularExpression:

    GENERAL_STRING = "^[A-Za-z_\s]+$"
    DATE = "^[0-9]{4}((-)[0-9]{2}){2}$"
    DATE_DATETIME = "^[0-9]{4}((-)[0-9]{2}){2}([\s]{1}[0-9]{2}((:)[0-9]{2}){2})?$"
    AZURE_DWH_KEY_VAULT_URI = "^(https:\/\/kv-ad-)((d)|(u)|(p))(-westeu-01\.vault\.azure\.net\/)$"
    AZURE_DWH_KEY_VAULT_SECRET_NAME = "^(sec)((-)[a-z0-9]+)+$"
    UUID = "^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$"
    # This regular expression was changed in this Notebook only (UNICORN_API_ENTITY_SCHEMA_NAME)
    UNICORN_API_ENTITY_SCHEMA_NAME = "^[a-z\_]+$"
    UNICORN_API_ENTITY_NAME = "^([A-Z]+[a-z]*)+$"
    UNICORN_API_ENTITY_ATTRIBUTE_NAME = "^([A-Z]{1}[a-z]+)+$"


In [ ]:
class AzureDWHKeyVaultHelper:

    _key_vault_uri = None

    @Decorator.validate_kwargs(schema={
        "type": "object",
        "properties": {
            "key_vault_uri": {
                "type": "string",
                "pattern": RegularExpression.AZURE_DWH_KEY_VAULT_URI
            },
        },
        "required": [
            "key_vault_uri",
        ],
        "additionalProperties": False
    })
    def __init__(self, key_vault_uri: str) -> None:
        self._key_vault_uri = key_vault_uri

    def __del__(self) -> None:
        pass

    @Decorator.validate_kwargs(schema={
        "type": "object",
        "properties": {
            "name": {
                "type": "string",
                "pattern": RegularExpression.AZURE_DWH_KEY_VAULT_SECRET_NAME
            },
        },
        "required": [
            "name",
        ],
        "additionalProperties": False
    })
    def get_secret_by_name(self, name: str) -> str:
        return getSecret(self._key_vault_uri, name)


In [ ]:
class MicrosoftFabricLakehouseHelper:

    @classmethod
    @Decorator.validate_kwargs(schema={
        "type": "object",
        "properties": {
            "workspace_id": {
                "type": "string",
                "pattern": RegularExpression.UUID
            },
            "lakehouse_id": {
                "type": "string",
                "pattern": RegularExpression.UUID
            }
        },
        "required": [
            "workspace_id",
            "lakehouse_id"
        ],
        "additionalProperties": False
    })
    def get_lakehouse_abfs_path(cls, workspace_id: str, lakehouse_id: str) -> str:
        return "abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/".format(
            workspace_id=workspace_id, lakehouse_id=lakehouse_id
        )

    @classmethod
    @Decorator.validate_kwargs(schema={
        "type": "object",
        "properties": {
            "workspace_id": {
                "type": "string",
                "pattern": RegularExpression.UUID
            },
            "lakehouse_id": {
                "type": "string",
                "pattern": RegularExpression.UUID
            },
            "entity_schema": {
                "type": "string",
                "pattern": RegularExpression.GENERAL_STRING
            },
            "entity": {
                "type": "string",
                "pattern": RegularExpression.GENERAL_STRING
            }
        },
        "required": [
            "workspace_id",
            "lakehouse_id",
            "entity_schema",
            "entity"
        ],
        "additionalProperties": False
    })
    def get_delta_table_abfs_path(cls, workspace_id: str, lakehouse_id: str, entity_schema: str, entity: str) -> str:
        return "{lh_abfs_path}Tables/{entity_schema}/{entity}".format(
            lh_abfs_path=cls.get_lakehouse_abfs_path(
                workspace_id=workspace_id, lakehouse_id=lakehouse_id
            ), entity_schema=entity_schema, entity=entity
        )

    @classmethod
    def get_delta_table_data_types(cls, delta_table: DeltaTable) -> dict:
        return {
            attribute_info.__dict__["name"]: attribute_info.__dict__["dataType"].__class__.__name__
            for attribute_info in delta_table.toDF().schema
        }


In [ ]:
class BronzeLayerDataFrameProcessor:

    _BRONZE_STAGE_DATAFRAME_TECH_COLUMNS = [
        "TechExecutorRunID",
        "TechProcessorRunID",
        "TechProcessingDateTime",
        "TechBusinessDateTime"
    ]

    _BRONZE_DATAFRAME_TECH_COLUMNS = [
        "TechExecutorRunID",
        "TechProcessorRunID",
        "TechProcessingDateTime",
        "TechBusinessDateTime"
    ]

    _BRONZE_HIST_DATAFRAME_TECH_COLUMNS = [
        "TechExecutorRunID",
        "TechProcessorRunID",
        "TechProcessingDateTime",
        "TechBusinessDateTime",
        "TechOperationType"
    ]

    _enable_processing_logging = None
    _processing_logs = None
    _processing_datetime = None
    _executor_run_id = None
    _business_datetime = None
    _bronze_stage_delta_table_path = None
    _bronze_delta_table_path = None
    _bronze_hist_delta_table_path = None
    _dataframe = None
    _primary_key = None
    _change_tracking_attribute = None
    _filter_pattern = None
    _filter_pattern_parameters = None
    _bronze_stage_delta_table_dataframe = None
    _bronze_delta_table_dataframe = None
    _primary_keys_to_load_to_bronze_hist = None
    _entity_processing_stats = None

    def __init__(self, processing_datetime: datetime, executor_run_id: str, business_datetime: datetime,
                 bronze_stage_delta_table_path: str, bronze_delta_table_path: str, bronze_hist_delta_table_path: str,
                 dataframe: DataFrame, primary_key: Union[str, list],
                 change_tracking_attribute: Union[None, str] = None, filter_pattern: Union[None, str] = None,
                 filter_pattern_parameters: [None, dict] = None) -> None:
        self._processing_datetime = processing_datetime.strftime("%Y-%m-%d %H:%M:%S")
        self._executor_run_id = executor_run_id
        self._business_datetime = business_datetime
        self._bronze_stage_delta_table_path = bronze_stage_delta_table_path
        self._bronze_delta_table_path = bronze_delta_table_path
        self._bronze_hist_delta_table_path = bronze_hist_delta_table_path
        self._dataframe = dataframe
        self._primary_key = primary_key if isinstance(primary_key, list) else [primary_key]
        self._change_tracking_attribute = change_tracking_attribute
        self._filter_pattern = filter_pattern
        self._filter_pattern_parameters = filter_pattern_parameters

    def __del__(self):
        pass

    def _enrich_dataframe_columns(self, columns: list, enrich_for: str) -> list:
        for column in reversed({
            "bronze_stage": self._BRONZE_STAGE_DATAFRAME_TECH_COLUMNS,
            "bronze": self._BRONZE_DATAFRAME_TECH_COLUMNS,
            "bronze_hist": self._BRONZE_HIST_DATAFRAME_TECH_COLUMNS
        }[enrich_for]):
            columns.insert(0, column)
        return columns

    def _enrich_dataframe(self, dataframe: DataFrame, enrich_for: str) -> DataFrame:
        if enrich_for in ["bronze_stage", "bronze"]:
            return dataframe.withColumns({
                "TechExecutorRunID": lit(self._executor_run_id).cast(StringType()),
                "TechProcessorRunID": lit(notebookutils.runtime.context["activityId"]).cast(StringType()),
                "TechProcessingDateTime": lit(self._processing_datetime).cast(TimestampType()),
                "TechBusinessDateTime": lit(self._business_datetime.strftime("%Y-%m-%d %H:%M:%S")).cast(TimestampType())
            })
        elif enrich_for == "bronze_hist":
            return dataframe.withColumns({
                "TechExecutorRunID": lit(self._executor_run_id).cast(StringType()),
                "TechProcessorRunID": lit(notebookutils.runtime.context["activityId"]).cast(StringType()),
                "TechProcessingDateTime": lit(self._processing_datetime).cast(TimestampType()),
                "TechBusinessDateTime": lit(
                    self._business_datetime.strftime("%Y-%m-%d %H:%M:%S")
                ).cast(TimestampType()),
                "TechOperationType": when(expr(str=" and ".join([
                    "TechPrimaryKeyBronzeStage_{primary_key_column} is null".format(
                        primary_key_column=primary_key_column
                    ) for primary_key_column in self._primary_key
                ])), "Delete").otherwise("Update")
            })
        elif enrich_for == "row_hash":
            dataframe = dataframe.withColumn("TechRowConcatenation", concat_ws("", *list(filter(
                lambda column: column not in (
                    self._BRONZE_STAGE_DATAFRAME_TECH_COLUMNS + self._BRONZE_DATAFRAME_TECH_COLUMNS
                ), dataframe.columns
            ))))
            dataframe = dataframe.withColumn("TechRowHash", sha2("TechRowConcatenation", 256))
            return dataframe.drop("TechRowConcatenation")

    def _add_processing_log(self, step_name: str, step_point: str, sub_step_name: Union[None, str] = None) -> None:
        if self._enable_processing_logging is True:
            if self._processing_logs is None:
                self._processing_logs = {}
            if "bronze_stage_delta_table_path" not in self._processing_logs.keys():
                 self._processing_logs["bronze_stage_delta_table_path"] = self._bronze_stage_delta_table_path
            if step_name not in self._processing_logs.keys():
                self._processing_logs[step_name] = {
                    "start": None,
                    "end": None,
                    "duration": None,
                    "sub_steps": {}
                }
            if sub_step_name is None:
                self._processing_logs[step_name][step_point] = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")
            else:
                if sub_step_name not in self._processing_logs[step_name]["sub_steps"].keys():
                    self._processing_logs[step_name]["sub_steps"][sub_step_name] = {
                        "start": None,
                        "end": None,
                        "duration": None
                    }
                self._processing_logs[step_name]["sub_steps"][sub_step_name][step_point] = \
                    datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")
            if step_point == "end":
                if sub_step_name is None:
                    self._processing_logs[step_name]["duration"] = (
                        datetime.strptime(self._processing_logs[step_name]["end"], "%Y-%m-%d %H:%M:%S.%f") -
                        datetime.strptime(self._processing_logs[step_name]["start"], "%Y-%m-%d %H:%M:%S.%f")
                    ).total_seconds()
                else:
                    self._processing_logs[step_name]["sub_steps"][sub_step_name]["duration"] = (
                        datetime.strptime(self._processing_logs[step_name]["sub_steps"][sub_step_name]["end"], "%Y-%m-%d %H:%M:%S.%f") -
                        datetime.strptime(self._processing_logs[step_name]["sub_steps"][sub_step_name]["start"], "%Y-%m-%d %H:%M:%S.%f")
                    ).total_seconds()

    def _get_bronze_stage_dataframe(self) -> None:
        self._add_processing_log(step_name="_get_bronze_stage_dataframe", step_point="start")
        columns = self._dataframe.columns
        dataframe = self._enrich_dataframe(
            dataframe=self._dataframe, enrich_for="bronze_stage"
        ).select(self._enrich_dataframe_columns(columns=columns, enrich_for="bronze_stage"))
        self._add_processing_log(step_name="_get_bronze_stage_dataframe", step_point="end")
        return dataframe

    def _load_dataframe_to_bronze_stage(self) -> None:
        self._add_processing_log(step_name="_load_dataframe_to_bronze_stage", step_point="start")
        bronze_stage_dataframe = self._get_bronze_stage_dataframe()
        bronze_stage_dataframe.write.format("delta").mode("overwrite").save(
            self._bronze_stage_delta_table_path
        )
        self._add_processing_log(step_name="_load_dataframe_to_bronze_stage", step_point="start", sub_step_name="vacuum")
        bronze_stage_delta_table = DeltaTable.forPath(sparkSession=spark, path=self._bronze_stage_delta_table_path)
        bronze_stage_delta_table.vacuum(retentionHours=3)
        self._entity_processing_stats = {
            "bronze_stg_loading_end_datetime_utc": datetime.now(timezone.utc),
            "bronze_stg_volume_mb": round(bronze_stage_delta_table.detail()[[
                "sizeInBytes"
            ]].collect()[0]["sizeInBytes"]/(1024 * 1024), 6)
        }
        self._add_processing_log(step_name="_load_dataframe_to_bronze_stage", step_point="end", sub_step_name="vacuum")
        self._add_processing_log(step_name="_load_dataframe_to_bronze_stage", step_point="end")

    def _get_dataframe_to_compare(self, layer: str, dataframe: DataFrame, extended_structure: bool) -> DataFrame:
        self._add_processing_log(
            step_name="_prepare_primary_keys_to_load_to_bronze_hist", step_point="start",
            sub_step_name="_get_dataframe_to_compare ({layer})".format(layer=layer)
        )
        dataframe = (
            self._enrich_dataframe(dataframe=dataframe, enrich_for="row_hash") if
            extended_structure is False and self._change_tracking_attribute is None else dataframe
        ).select(
            *(
                [
                    col(primary_key_column).alias("TechPrimaryKey{layer}_{primary_key_column}".format(
                        layer=layer, primary_key_column=primary_key_column
                    )) for primary_key_column in self._primary_key
                ]
            ),
            *(
                [
                    col("TechRowHash").alias("TechRowHash{layer}".format(layer=layer))
                ] if extended_structure is False and self._change_tracking_attribute is None else []
            ),
            *(
                [
                    col(self._change_tracking_attribute).alias("TechChangeTrackingAttribute{layer}".format(layer=layer))
                ] if extended_structure is False and self._change_tracking_attribute is not None else []
            )
        )
        self._add_processing_log(
            step_name="_prepare_primary_keys_to_load_to_bronze_hist", step_point="end",
            sub_step_name="_get_dataframe_to_compare ({layer})".format(layer=layer)
        )
        return dataframe

    def _prepare_primary_keys_to_load_to_bronze_hist(self) -> None:
        self._add_processing_log(step_name="_prepare_primary_keys_to_load_to_bronze_hist", step_point="start")
        extended_structure = len(self._bronze_stage_delta_table_dataframe.columns) > len(list(filter(
            lambda column: column not in self._BRONZE_DATAFRAME_TECH_COLUMNS,
            self._bronze_delta_table_dataframe.columns
        )))
        self._primary_keys_to_load_to_bronze_hist = self._get_dataframe_to_compare(
            layer="BronzeStage", dataframe=self._bronze_stage_delta_table_dataframe,
            extended_structure=extended_structure
        ).alias("bronze_stage_dataframe_to_compare").join(
            other=self._get_dataframe_to_compare(
                layer="Bronze", dataframe=self._bronze_delta_table_dataframe,
                extended_structure=extended_structure
            ).alias("bronze_dataframe_to_compare"), on=expr(str=" and ".join([
                "bronze_stage_dataframe_to_compare.TechPrimaryKeyBronzeStage_{primary_key_column} == "
                "bronze_dataframe_to_compare.TechPrimaryKeyBronze_{primary_key_column}".format(
                    primary_key_column=primary_key_column
                ) for primary_key_column in self._primary_key
            ])), how="full"
        ).select(*[
            "{prefix}_{primary_key_column}".format(prefix=prefix, primary_key_column=primary_key_column)
            for prefix in ["TechPrimaryKeyBronze", "TechPrimaryKeyBronzeStage"]
            for primary_key_column in self._primary_key
        ]).where(
            (" or ".join([
                "({primary_key_condition})".format(primary_key_condition=" and ".join([
                    "TechPrimaryKeyBronzeStage_{primary_key_column} is null".format(
                        primary_key_column=primary_key_column
                    ) for primary_key_column in self._primary_key
                ])), "TechRowHashBronzeStage != TechRowHashBronze"
            ])) if extended_structure is False and self._change_tracking_attribute is None else (
                "1 == 1" if extended_structure is True else (" or ".join([
                    "({primary_key_condition})".format(primary_key_condition=" and ".join([
                        "TechPrimaryKeyBronzeStage_{primary_key_column} is null".format(
                            primary_key_column=primary_key_column
                        ) for primary_key_column in self._primary_key
                    ])), "TechChangeTrackingAttributeBronzeStage != TechChangeTrackingAttributeBronze"
                ]))
            )
        )
        self._add_processing_log(step_name="_prepare_primary_keys_to_load_to_bronze_hist", step_point="end")

    def _get_dataframe_to_load_to_bronze_hist(self) -> DataFrame:
        self._add_processing_log(step_name="_get_dataframe_to_load_to_bronze_hist", step_point="start")
        dataframe = self._bronze_delta_table_dataframe.alias("bronze_delta_table_dataframe").join(
            other=self._primary_keys_to_load_to_bronze_hist.alias("primary_keys_to_load_to_bronze_hist"),
            on=expr(str=" and ".join([
                "bronze_delta_table_dataframe.{primary_key_column} = "
                "primary_keys_to_load_to_bronze_hist.TechPrimaryKeyBronze_{primary_key_column}".format(
                    primary_key_column=primary_key_column
                ) for primary_key_column in self._primary_key
            ])), how="inner"
        ).drop(*[
            "TechPrimaryKeyBronze_{primary_key_column}".format(primary_key_column=primary_key_column)
            for primary_key_column in self._primary_key
        ]).withColumnRenamed(
            "TechExecutorRunID", "TechExecutorRunIDOriginal"
        ).withColumnRenamed(
            "TechProcessorRunID", "TechProcessorRunIDOriginal"
        ).withColumnRenamed(
            "TechProcessingDateTime", "TechProcessingDateTimeOriginal"
        ).withColumnRenamed(
            "TechBusinessDateTime", "TechBusinessDateTimeOriginal"
        )
        self._add_processing_log(step_name="_get_dataframe_to_load_to_bronze_hist", step_point="end")
        return dataframe

    def _prepare_dataframe_for_loading_to_bronze_hist(self, dataframe: DataFrame) -> DataFrame:
        self._add_processing_log(step_name="_prepare_dataframe_for_loading_to_bronze_hist", step_point="start")
        dataframe_columns = dataframe.columns
        dataframe = self._enrich_dataframe(
            dataframe=dataframe, enrich_for="bronze_hist"
        ).select(
            self._enrich_dataframe_columns(columns=dataframe_columns, enrich_for="bronze_hist")
        ).drop(*[
            "TechPrimaryKeyBronzeStage_{primary_key_column}".format(primary_key_column=primary_key_column)
            for primary_key_column in self._primary_key
        ])
        self._add_processing_log(step_name="_prepare_dataframe_for_loading_to_bronze_hist", step_point="end")
        return dataframe

    def _load_dataframe_to_bronze_hist(self) -> None:
        self._add_processing_log(step_name="_load_dataframe_to_bronze_hist", step_point="start")
        dataframe = self._get_dataframe_to_load_to_bronze_hist()
        self._add_processing_log(step_name="_load_dataframe_to_bronze_hist", step_point="start", sub_step_name="counting")
        dataframe_count = self._primary_keys_to_load_to_bronze_hist.count()
        self._add_processing_log(step_name="_load_dataframe_to_bronze_hist", step_point="end", sub_step_name="counting")
        if dataframe_count > 0:
            df = self._prepare_dataframe_for_loading_to_bronze_hist(
                dataframe=dataframe
            )
            self._add_processing_log(step_name="_load_dataframe_to_bronze_hist", step_point="start", sub_step_name="write")
            df.write.format("delta").mode(
                "append" if DeltaTable.isDeltaTable(
                    sparkSession=spark, identifier=self._bronze_hist_delta_table_path
                ) is True else "overwrite"
            ).save(self._bronze_hist_delta_table_path)
            self._add_processing_log(step_name="_load_dataframe_to_bronze_hist", step_point="end", sub_step_name="write")
            self._add_processing_log(step_name="_load_dataframe_to_bronze_hist", step_point="start", sub_step_name="vacuum")
            DeltaTable.forPath(
                sparkSession=spark, path=self._bronze_hist_delta_table_path
            ).vacuum(retentionHours=3)
            self._add_processing_log(step_name="_load_dataframe_to_bronze_hist", step_point="end", sub_step_name="vacuum")
        self._add_processing_log(step_name="_load_dataframe_to_bronze_hist", step_point="end")

    def _actualize_bronze_entity(self, bronze_stage_delta_table_dataframe: DataFrame,
                                 bronze_delta_table: DeltaTable) -> None:
        self._add_processing_log(step_name="_actualize_bronze_entity", step_point="start")
        bronze_stage_delta_table_dataframe = bronze_stage_delta_table_dataframe.alias(
            "bronze_stage_delta_table_dataframe"
        ).join(
            other=self._primary_keys_to_load_to_bronze_hist.where(
                expr(str=" and ".join([
                    "TechPrimaryKeyBronzeStage_{primary_key_column} is not null".format(
                        primary_key_column=primary_key_column
                    ) for primary_key_column in self._primary_key
                ]))
            ).alias("updated_primary_keys"), on=expr(str=" and ".join([
                "bronze_stage_delta_table_dataframe.{primary_key_column} = "
                "updated_primary_keys.TechPrimaryKeyBronzeStage_{primary_key_column}".format(
                    primary_key_column=primary_key_column
                ) for primary_key_column in self._primary_key
            ])), how="left"
        )
        bronze_stage_delta_table_dataframe_columns = bronze_stage_delta_table_dataframe.columns
        bronze_delta_table.alias("bronze").merge(
            source=bronze_stage_delta_table_dataframe.alias("bronze_stage"),
            condition=" and ".join(
                ([] if (self._filter_pattern is None or self._filter_pattern_parameters is None) else [
                    self._filter_pattern.format(**(self._filter_pattern_parameters | {
                        "table_alias": "bronze."
                    }))
                ]) + [
                    "bronze.{primary_key_column} == bronze_stage.{primary_key_column}".format(
                        primary_key_column=primary_key_column
                    ) for primary_key_column in self._primary_key
                ]
            )
        ).whenMatchedUpdate(set={
            column: "bronze_stage.{column}".format(column=column)
            for column in bronze_stage_delta_table_dataframe_columns
            if column not in [
                "{prefix}_{primary_key_column}".format(prefix=prefix, primary_key_column=primary_key_column)
                for prefix in ["TechPrimaryKeyBronze", "TechPrimaryKeyBronzeStage"]
                for primary_key_column in self._primary_key
            ]
        }, condition=" and ".join([
            "bronze_stage.TechPrimaryKeyBronzeStage_{primary_key_column} is not null".format(
                primary_key_column=primary_key_column
            ) for primary_key_column in self._primary_key
        ])).whenNotMatchedInsert(values={
            column: "bronze_stage.{column}".format(column=column)
            for column in bronze_stage_delta_table_dataframe_columns
            if column not in [
                "{prefix}_{primary_key_column}".format(prefix=prefix, primary_key_column=primary_key_column)
                for prefix in ["TechPrimaryKeyBronze", "TechPrimaryKeyBronzeStage"]
                for primary_key_column in self._primary_key
            ]
        }).whenNotMatchedBySourceDelete(
            condition=("1 == 1" if (self._filter_pattern is None or self._filter_pattern_parameters is None) else
                self._filter_pattern.format(**(self._filter_pattern_parameters | {
                    "table_alias": "bronze."
                }))
            )
        ).execute()
        self._add_processing_log(step_name="_actualize_bronze_entity", step_point="end")

    def _load_dataframe_to_bronze(self) -> None:
        self._add_processing_log(step_name="_load_dataframe_to_bronze", step_point="start")
        self._bronze_stage_delta_table_dataframe = DeltaTable.forPath(
            sparkSession=spark, path=self._bronze_stage_delta_table_path
        ).toDF().drop(*self._BRONZE_STAGE_DATAFRAME_TECH_COLUMNS)
        self._entity_processing_stats["bronze_stg_records_count"] = self._bronze_stage_delta_table_dataframe.count()
        if DeltaTable.isDeltaTable(sparkSession=spark, identifier=self._bronze_delta_table_path) is True:
            bronze_delta_table = DeltaTable.forPath(sparkSession=spark, path=self._bronze_delta_table_path)
            self._bronze_delta_table_dataframe = bronze_delta_table.toDF() if (
                self._filter_pattern is None or self._filter_pattern_parameters is None
            ) else bronze_delta_table.toDF().where(self._filter_pattern.format(**(self._filter_pattern_parameters | {
                "table_alias": ""
            })))
            self._prepare_primary_keys_to_load_to_bronze_hist()
            self._load_dataframe_to_bronze_hist()
            self._actualize_bronze_entity(
                bronze_stage_delta_table_dataframe=self._enrich_dataframe(
                    dataframe=self._bronze_stage_delta_table_dataframe, enrich_for="bronze"
                ).select(self._enrich_dataframe_columns(
                    columns=self._bronze_stage_delta_table_dataframe.columns, enrich_for="bronze"
                )), bronze_delta_table=bronze_delta_table
            )
            bronze_delta_table.vacuum(retentionHours=3)
        else:
            self._enrich_dataframe(
                dataframe=self._bronze_stage_delta_table_dataframe, enrich_for="bronze"
            ).select(self._enrich_dataframe_columns(
                columns=self._bronze_stage_delta_table_dataframe.columns, enrich_for="bronze"
            )).write.format("delta").mode("overwrite").save(self._bronze_delta_table_path)
        self._add_processing_log(step_name="_load_dataframe_to_bronze", step_point="end")

    def process(self) -> dict:
        self._enable_processing_logging = True
        self._add_processing_log(step_name="process", step_point="start")
        self._load_dataframe_to_bronze_stage()
        self._dataframe = None
        self._load_dataframe_to_bronze()
        self._add_processing_log(step_name="process", step_point="end")
        if isinstance(self._processing_logs, dict):
            print(dumps(obj=self._processing_logs, ensure_ascii=False, indent=4))
        return self._entity_processing_stats


In [ ]:
class OAuthToken:

    _OAUTH_TOKEN_REQUEST_URL_TEMPLATE = "https://login.microsoftonline.com/{tenant_id}/oauth2/token"

    @classmethod
    def _prepare_oauth_token(cls, oauth_token_response_dict: dict) -> str:
        return "{token_type} {access_token}".format(
            token_type=oauth_token_response_dict["token_type"], access_token=oauth_token_response_dict["access_token"]
        )

    @classmethod
    def get_oauth_token_with_client_credentials(cls, tenant_id: str, client_id: str, client_secret: str,
                                                resource: str, return_prepared: bool = True) -> Response:
        oauth_token_response = post(url=cls._OAUTH_TOKEN_REQUEST_URL_TEMPLATE.format(tenant_id=tenant_id),
                                    headers={
                                        "Content-type": "application/x-www-form-urlencoded"
                                    },
                                    data={
                                        "grant_type": "client_credentials",
                                        "client_id": client_id,
                                        "client_secret": client_secret,
                                        "resource": resource
                                    })
        if oauth_token_response.status_code == 200:
            if return_prepared is True:
                return cls._prepare_oauth_token(oauth_token_response_dict=oauth_token_response.json())
            return oauth_token_response
        raise ValueError("Couldn't retrieve OAuth token, details - [{details}]".
                         format(details=oauth_token_response.text))


class MicrosoftFabricRESTAPIV1:

    @classmethod
    def get_workspace_sql_databases_response(cls, workspace_id: str, access_token: str) -> Response:
        workspace_sql_databases_response = get(
            url="https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/sqlDatabases".format(
                workspace_id=workspace_id
            ), headers={
                "Authorization": access_token
            }
        )
        if workspace_sql_databases_response.status_code == 200:
            return workspace_sql_databases_response
        raise Exception(
            "Something went wrong while retrieving SQL Databases for Microsoft Fabric Workspace with ID - "
            "[{workspace_id}]: response status code - [{response_status_code}], response - [{response}].".format(
                workspace_id=workspace_id, response_status_code=workspace_sql_databases_response.status_code,
                response=workspace_sql_databases_response.text
            )
        )


class MicrosoftFabricSQLDB:

    _SERVICE_PRINCIPAL_CONNECTION_STRING_TEMPLATE = \
        "driver=ODBC Driver 18 for SQL Server;server={server};database={database};UID={client_id};" \
        "PWD={client_secret};Authentication=ActiveDirectoryServicePrincipal;Encrypt=yes;" \
        "TrustServerCertificate=no;"

    _connection = None

    def __init__(self, server: str, database: str, auth_type: str, credentials: dict) -> None:
        print("Trying to connect to Fabric SQL DB with the following parameters: server - [{server}], "
              "database - [{database}].".format(server=server, database=database))
        connection_established, maximum_tries, current_try, connection_establishing_exception = False, 3, 0, None
        while connection_established is False and current_try < maximum_tries:
            try:
                current_try += 1
                print("Fabric SQL DB connection attempt number - [{current_try}].".format(current_try=current_try))
                getattr(self, {
                    "service_principal": "_init_service_principal_connection"
                }[auth_type])(server=server, database=database, credentials=credentials)
                connection_established = True
                print("Fabric SQL DB connection established successfully.")
            except Exception as e:
                connection_establishing_exception = e
                print("Starting sleep for 5 seconds.")
                sleep(5)
                pass
        if connection_established is False:
            raise connection_establishing_exception

    def __del__(self) -> None:
        if self._connection is not None:
            try:
                self._connection.close()
            except Exception:
                pass

    def _init_service_principal_connection(self, server: str, database: str, credentials: dict) -> None:
        self._connection = connect(self._SERVICE_PRINCIPAL_CONNECTION_STRING_TEMPLATE.format(
            server=server, database=database, client_id=credentials["client_id"],
            client_secret=credentials["client_secret"]
        ))

    def execute_query(self, query: str, commit: bool = False, fetch_results: bool = True) -> None:
        cursor = self._connection.cursor()
        cursor.execute(query)
        if commit is True:
            cursor.commit()
        if fetch_results is True:
            cursor_results = cursor.fetchall()
        cursor.close()
        if fetch_results is True:
            return cursor_results

    def close_connection(self) -> None:
        self._connection.close()
        self._connection = None


class EntitiesProcessingsStats:

    _ENTITIES_PROCESSING_STATS_INSERT_STATEMENT_TEMPLATE = """
        insert into [etl].[entities_processings_stats] (
            [created_by],
            [updated_by],
            [business_datetime],
            [stats]
        )
        select N'{executor}' as [created_by],
            N'{executor}' as [updated_by],
            N'{business_datetime}' as [business_datetime],
            N'{stats}' as [stats]

    """

    @classmethod
    def insert_entities_processing_stats(cls, client_id: str, client_secret: str, executor: str,
                                         business_datetime: dict, stats: dict) -> None:
        sql_db_etl_info = list(filter(
            lambda sql_db_info: sql_db_info["displayName"] == "sql-db-etl",
            MicrosoftFabricRESTAPIV1.get_workspace_sql_databases_response(
                workspace_id=notebookutils.runtime.context["currentWorkspaceId"],
                access_token=OAuthToken.get_oauth_token_with_client_credentials(
                    tenant_id=spark.conf.get("trident.tenant.id"), client_id=client_id,
                    client_secret=client_secret, resource="https://api.fabric.microsoft.com/"
                )
            ).json()["value"]
        ))[0]
        MicrosoftFabricSQLDB(
            server=sql_db_etl_info["properties"]["serverFqdn"].split(",")[0],
            database=sql_db_etl_info["properties"]["databaseName"], auth_type="service_principal",
            credentials={
                "client_id": client_id,
                "client_secret": client_secret
            }
        ).execute_query(query=cls._ENTITIES_PROCESSING_STATS_INSERT_STATEMENT_TEMPLATE.format(
            executor=executor, business_datetime=business_datetime, stats=dumps(obj=stats, ensure_ascii=False)
        ), commit=True, fetch_results=False)


In [ ]:
class UnicornAPICustomEntitiesProcessor:

    _executor_run_id = None
    _business_datetime = None
    _key_vault_uri = None
    _workspace_id = None
    _lh_bronze_stage_id = None
    _lh_bronze_id = None
    _lh_bronze_hist_id = None
    _entities_config = None
    _azure_dwh_key_vault_helper_instance = None
    _api_session = None
    _bronze_stg_entities_processing_stats = None

    def __init__(self) -> None:
        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", True)
        spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", False)
        spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
        spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

    def __del__(self) -> None:
        if self._api_session is not None:
            self._api_session.close()

    @classmethod
    def _get_prepared_business_datetime(cls, business_datetime: str) -> datetime:
        if fullmatch(RegularExpression.DATE, business_datetime) is None:
            return datetime.strptime(business_datetime, "%Y-%m-%d %H:%M:%S")
        return datetime.strptime(business_datetime, "%Y-%m-%d")

    def _init_lakehouses_metadata(self) -> None:
        [
            setattr(self, {
                "lhbronzestgad": "_lh_bronze_stage_id",
                "lhbronzead": "_lh_bronze_id",
                "lhbronzehistad": "_lh_bronze_hist_id"
            }[lakehouse["displayName"]], lakehouse["id"])
            for lakehouse in notebookutils.lakehouse.list(self._workspace_id)
            if lakehouse["displayName"] in [
                "lhbronzestgad",
                "lhbronzead",
                "lhbronzehistad"
            ]
        ]
        for lakehouse_id in [
            self._lh_bronze_stage_id,
            self._lh_bronze_id,
            self._lh_bronze_hist_id
        ]:
            if lakehouse_id is None:
                raise Exception(
                    "Lakehouses metadata initialization error: lakehouse ID cannot be None "
                    "(bronze stage lakehouse ID - [{lh_bronze_stage_id}], bronze lakehouse ID - "
                    "[{lh_bronze_id}], bronze hist lakehouse ID - [{lh_bronze_hist_id})].".format(
                        lh_bronze_stage_id=self._lh_bronze_stage_id, lh_bronze_id=self._lh_bronze_id,
                        lh_bronze_hist_id=self._lh_bronze_hist_id
                    )
                )

    @classmethod
    def _get_api_url(cls, url_part: str) -> str:
        return "https://plc.creatio.com/{url_part}".format(url_part=url_part)

    def _init_api_session(self) -> None:
        self._azure_dwh_key_vault_helper_instance = AzureDWHKeyVaultHelper(key_vault_uri=self._key_vault_uri)
        self._api_session = Session()
        api_login_response = self._api_session.post(
            url=self._get_api_url(url_part="ServiceModel/AuthService.svc/Login"),
            headers={
                "Content-Type": "application/json"
            },
            data=dumps(obj={
                "UserName": self._azure_dwh_key_vault_helper_instance.get_secret_by_name(
                    name="sec-unicorn-api-user-name"
                ),
                "UserPassword": self._azure_dwh_key_vault_helper_instance.get_secret_by_name(
                    name="sec-unicorn-api-user-pass"
                )
            }, ensure_ascii=False)
        )
        if api_login_response.json()["Code"] != 0:
            raise Exception(
                "Something went wrong during Unicorn API login process, error details - [{details}].".format(
                    details=api_login_response.text
                )
            )

    def _close_api_session(self) -> None:
        self._api_session.close()
        self._api_session = None

    def _get_entity_raw_dataframe_info(self, entity_schema: str, entity: str,
                                       bronze_stage_delta_table_path: str) -> DataFrame:
        columns, structure = [], []
        for entity_attribute, entity_attribute_config in self._entities_config[entity_schema][entity]["spark_schema"].items():
            columns.append(entity_attribute)
            structure.append(StructField(
                name=entity_attribute, dataType=getattr(PySparkSQLTypes, entity_attribute_config["data_type"])(
                    **(entity_attribute_config["data_type_config"] if "data_type_config" in entity_attribute_config.keys() else {})
                ), nullable=entity_attribute_config["nullable"]
            ))
        entity_api_data, entity_api_data_retrieved_successfully, entity_api_data_retrieving_errors = None, False, []
        entity_api_data_retrieving_max_attempts_number, entity_api_data_retrieving_attempt_number = 3, 0
        start_datetime_utc = None
        while (entity_api_data_retrieved_successfully is False and
               entity_api_data_retrieving_attempt_number < entity_api_data_retrieving_max_attempts_number):
            start_datetime_utc = datetime.now(timezone.utc)
            entity_api_data_retrieving_attempt_number += 1
            entity_api_data_response = None
            try:
                entity_api_data_response = getattr(
                    self._api_session, self._entities_config[entity_schema][entity]["api_http_request_method"].lower()
                )(url=self._get_api_url(
                    url_part=self._entities_config[entity_schema][entity]["api_endpoint_route"]
                ))
                entity_api_data = entity_api_data_response.json()
                for api_response_path_item in self._entities_config[entity_schema][entity]["api_response_path"].split("|"):
                    entity_api_data = entity_api_data[api_response_path_item]
                entity_api_data_retrieved_successfully = True
            except Exception as e:
                error_type = e.__class__.__name__
                entity_api_data_retrieving_errors.append({
                    "attempt_number": entity_api_data_retrieving_attempt_number,
                    "error_type": error_type,
                    "error": str(e),
                    **({
                        "response_status_code": entity_api_data_response.status_code
                    } if error_type == "KeyError" else {}),
                    **({
                        "response": entity_api_data_response.text
                    } if error_type == "KeyError" else {})
                })
                sleep(10)
        if entity_api_data_retrieved_successfully is False:
            raise Exception(entity_api_data_retrieving_errors)
        if len(entity_api_data) > 0:
            entity_api_columns = entity_api_data[0].keys()
            entity_spark_columns = self._entities_config[entity_schema][entity]["spark_schema"].keys()
            if len(set(entity_api_columns) - set(entity_spark_columns)) > 0 or \
                len(set(entity_spark_columns) - set(entity_api_columns)) > 0:
                raise Exception(
                    "Entity API/Spark structures mismatch: API structure - {entity_api_columns}, "
                    "Spark structure - {entity_spark_columns}.".format(
                        entity_api_columns=list(entity_api_columns), entity_spark_columns=list(entity_spark_columns)
                    )
                )
        return {
            "start_datetime_utc": start_datetime_utc,
            "raw_dataframe": spark.createDataFrame(
                data=entity_api_data, schema=StructType(structure)
            ).select(*columns)
        }

    def _get_delta_table_path(self, entity_schema: str, entity: str, layer: str) -> str:
        return MicrosoftFabricLakehouseHelper.get_delta_table_abfs_path(
            workspace_id=self._workspace_id, lakehouse_id={
                "bronze_stage": self._lh_bronze_stage_id,
                "bronze": self._lh_bronze_id,
                "bronze_hist": self._lh_bronze_hist_id
            }[layer], entity_schema="unicorn_{entity_schema}".format(entity_schema=entity_schema), entity=entity
        )

    def _process_entity_bronze_full_load(self, processing_datetime: datetime, entity_schema: str, entity: str) -> None:
        bronze_stage_delta_table_path = self._get_delta_table_path(
            entity_schema=entity_schema, entity=entity, layer="bronze_stage"
        )
        entity_raw_dataframe_info = self._get_entity_raw_dataframe_info(
            entity_schema=entity_schema, entity=entity,
            bronze_stage_delta_table_path=bronze_stage_delta_table_path
        )
        entity_processing_stats = BronzeLayerDataFrameProcessor(
            processing_datetime=processing_datetime,
            executor_run_id=self._executor_run_id, business_datetime=self._business_datetime,
            bronze_stage_delta_table_path=bronze_stage_delta_table_path,
            bronze_delta_table_path=self._get_delta_table_path(
                entity_schema=entity_schema, entity=entity, layer="bronze"
            ),
            bronze_hist_delta_table_path=self._get_delta_table_path(
                entity_schema=entity_schema, entity=entity, layer="bronze_hist"
            ),
            dataframe=entity_raw_dataframe_info["raw_dataframe"],
            primary_key=self._entities_config[entity_schema][entity]["primary_key"],
            change_tracking_attribute=(
                self._entities_config[entity_schema][entity]["change_tracking_attribute"] if
                "change_tracking_attribute" in self._entities_config[entity_schema][entity].keys() else None
            )
        ).process()
        if self._bronze_stg_entities_processing_stats is None:
            self._bronze_stg_entities_processing_stats = {}
        if entity_schema not in self._bronze_stg_entities_processing_stats.keys():
            self._bronze_stg_entities_processing_stats[entity_schema] = []
        self._bronze_stg_entities_processing_stats[entity_schema].append({
            "entityName": entity,
            "startDateTimeUTC": entity_raw_dataframe_info["start_datetime_utc"].strftime("%Y-%m-%d %H:%M:%S"),
            "endDateTimeUTC": entity_processing_stats[
                "bronze_stg_loading_end_datetime_utc"
            ].strftime("%Y-%m-%d %H:%M:%S"),
            "targetDataVolumeMB": entity_processing_stats["bronze_stg_volume_mb"],
            "recordsCount": entity_processing_stats["bronze_stg_records_count"]
        })

    @Decorator.validate_kwargs(schema={
        "type": "object",
        "properties": {
            "executor_run_id": {
                "type": "string",
                "minLength": 1
            },
            "business_datetime": {
                "type": "string",
                "pattern": RegularExpression.DATE_DATETIME
            },
            "key_vault_uri": {
                "type": "string",
                "pattern": RegularExpression.AZURE_DWH_KEY_VAULT_URI
            },
            "entities_config": {
                "type": "object",
                "patternProperties": {
                    RegularExpression.UNICORN_API_ENTITY_SCHEMA_NAME: {
                        "type": "object",
                        "patternProperties": {
                            RegularExpression.UNICORN_API_ENTITY_NAME: {
                                "type": "object",
                                "properties": {
                                    "load_type": {
                                        "type": "string",
                                        "enum": [
                                            "full"
                                        ]
                                    },
                                    "api_endpoint_route": {
                                        "type": "string",
                                        "minLength": 1
                                    },
                                    "api_http_request_method": {
                                        "type": "string",
                                        "enum": [
                                            "GET"
                                        ]
                                    },
                                    "api_response_path": {
                                        "type": "string",
                                        "minLength": 1
                                    },
                                    "primary_key": {
                                        "type": "array",
                                        "items": {
                                            "type": "string",
                                            "pattern": RegularExpression.UNICORN_API_ENTITY_ATTRIBUTE_NAME
                                        }
                                    },
                                    "spark_schema": {
                                        "type": "object",
                                        "patternProperties": {
                                            RegularExpression.UNICORN_API_ENTITY_ATTRIBUTE_NAME: {
                                                "type": "object",
                                                "properties": {
                                                    "data_type": {
                                                        "type": "string",
                                                        "minLength": 1
                                                    },
                                                    "data_type_config": {
                                                        "type": "object",
                                                        "minProperties": 1
                                                    },
                                                    "nullable": {
                                                        "type": "boolean"
                                                    }
                                                },
                                                "required": [
                                                    "data_type",
                                                    "nullable"
                                                ],
                                                "additionalProperties": False
                                            }
                                        }
                                    }
                                },
                                "required": [
                                    "load_type",
                                    "api_endpoint_route",
                                    "api_http_request_method",
                                    "api_response_path",
                                    "primary_key",
                                    "spark_schema"
                                ],
                                "additionalProperties": False
                            }
                        },
                        "additionalProperties": False
                    }
                },
                "additionalProperties": False
            }
        },
        "required": [
            "executor_run_id",
            "business_datetime",
            "key_vault_uri",
            "entities_config"
        ],
        "additionalProperties": False
    })
    def process_bronze_load(self, executor_run_id: str, business_datetime: str,
                            key_vault_uri: str, entities_config: dict):
        self._executor_run_id = executor_run_id
        self._business_datetime = self._get_prepared_business_datetime(business_datetime=business_datetime)
        self._key_vault_uri = key_vault_uri
        self._workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
        self._init_lakehouses_metadata()
        self._entities_config = entities_config
        self._init_api_session()
        processing_datetime, processing_errors = datetime.utcnow(), []
        for entities_schema, entities in self._entities_config.items():
            for entity, entity_config in entities.items():
                if entity_config["load_type"] == "full":
                    try:
                        self._process_entity_bronze_full_load(
                            processing_datetime=processing_datetime, entity_schema=entities_schema,
                            entity=entity
                        )
                    except Exception as e:
                        processing_errors.append(
                            "Failed to process entity: entities_schema - [{entities_schema}], "
                            "entity - [{entity}], error - [{error}].".format(
                                entities_schema=entities_schema, entity=entity, error=str(e)
                            )
                        )
        try:
            self._close_api_session()
        except Exception as e:
            processing_errors.append(str(e))
        if isinstance(self._bronze_stg_entities_processing_stats, dict):
            try:
                EntitiesProcessingsStats.insert_entities_processing_stats(
                    client_id=getSecret(self._key_vault_uri, "sec-appr-fbrcw-tech-user-client-id"),
                    client_secret = getSecret(self._key_vault_uri, "sec-appr-fbrcw-tech-user-secret"),
                    executor=executor_run_id, business_datetime=business_datetime,
                    stats=[
                        {
                            "step": "bronzeStg",
                            "bySystem": [
                                {
                                    "systemName": "Unicorn",
                                    "bySchema": [
                                        {
                                            "schemaName": schema,
                                            "byEntity": bronze_stg_entity_processing_stats
                                        } for schema, bronze_stg_entity_processing_stats in
                                        self._bronze_stg_entities_processing_stats.items()
                                    ]
                                }
                            ]
                        }
                    ]
                )
            except Exception as e:
                processing_errors.append(str(e))
        if len(processing_errors) > 0:
            raise Exception(processing_errors)


In [ ]:
params = loads(s=params)

getattr(UnicornAPICustomEntitiesProcessor(), params["process"])(
    **loads(params["tech_params"]), **loads(params["process_params"])
)